# VLM Models

**Module:** 15 — VLMs & Multimodal

OpenAI and the broader VLM landscape — strengths, gaps, and selection rubrics.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Compare hosted and open VLM offerings at product level
- Identify specialized vs generalist models
- Apply quality/cost/latency/privacy/region rubrics
- Route easy vs hard images across models


## Landscape Overview

### Definition
Frontier hosted APIs, regional clouds, and open weights you can self-host.

### Why it matters
Lock-in, residency, and cost curves differ more than demo quality suggests.

### How it works
Evaluate on *your* images; track limits, video support, structured outputs.

### Intuition
Pick for your workload distribution, not leaderboard screenshots.

### Pitfalls
- Text-only leaderboard shopping
- Ignoring retention/training-use policies

### When to use
Procurement and bakeoffs.


### Provider snapshot (verify current docs)

| Vendor | Examples | Notes |
|--------|----------|-------|
| OpenAI | GPT-4o / vision family | Tools + detail knobs |
| Google | Gemini multimodal | Long context |
| Anthropic | Claude vision | Docs/UI careful analysis |
| Meta | Llama vision | Open ecosystem |
| Alibaba | Qwen-VL | Strong open/commercial options |
| Open | LLaVA, InternVL, Florence-class | Self-host |
| Specialized | Donut, GOT-OCR, UI models | Narrow excellence |

```mermaid
flowchart LR
  W[Workload] --> N{Needs}
  N -->|retrieval| E[Dual-encoder]
  N -->|doc JSON| D[Doc VLM + OCR]
  N -->|chat| F[Frontier hosted]
  N -->|airgap| O[Open weights]
```


In [ ]:
# Demo 1: selection rubric
from dataclasses import dataclass

@dataclass
class ModelScore:
    name: str; quality: float; cost: float; latency: float; privacy: float; doc_skill: float
    def score(self, w):
        return (w["quality"]*self.quality + w["privacy"]*self.privacy + w["doc"]*self.doc_skill
                - w["cost"]*self.cost - w["latency"]*self.latency)

models = [
    ModelScore("hosted-frontier",0.9,0.8,0.5,0.5,0.85),
    ModelScore("hosted-mini",0.7,0.2,0.3,0.5,0.6),
    ModelScore("selfhost-open",0.72,0.3,0.7,0.95,0.7),
]
w = {"quality":0.35,"cost":0.2,"latency":0.15,"privacy":0.15,"doc":0.15}
print(sorted(((m.score(w), m.name) for m in models), reverse=True))


In [ ]:
# Demo 2: OpenAI vision request builder
import os, json
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "YOUR_OPENAI_API_KEY")

def build_openai_vision(model, url, prompt, detail="auto"):
    return {"model": model, "messages":[{"role":"user","content":[
        {"type":"text","text":prompt},
        {"type":"image_url","image_url":{"url":url,"detail":detail}},
    ]}], "max_tokens": 300}

print(json.dumps(build_openai_vision("gpt-4o-mini","https://example.com/ui.png","List UI bugs.","high"), indent=2))
print("live key?", OPENAI_API_KEY != "YOUR_OPENAI_API_KEY")


## Selection Tips & Routing

### Definition
Multi-objective optimization under policy constraints — not a single best model.

### Why it matters
Frontier for every thumbnail wastes money; mini for contracts wastes trust.

### How it works
Route by doc vs photo, boxes needed, language, hard_OCR, tenancy; escalate on validator fail.

### Intuition
A mini model on a good crop often beats frontier on a blurry full page.

### Pitfalls
- One global model string
- No frozen bakeoff set

### When to use
All production multimodal platforms.


In [ ]:
# Demo 3: router
def route_model(job):
    if job.get("residency")=="eu_only": return "eu-regional-vlm"
    if job.get("air_gapped"): return "selfhost-open-vlm"
    if job.get("task")=="embed": return "clip-siglip"
    if job.get("hard_ocr") or job.get("page_count",1)>3: return "frontier-vlm+ocr-hybrid"
    return "hosted-mini-vlm"

for j in [{"task":"alt_text"},{"task":"invoice","hard_ocr":True},{"task":"chat","residency":"eu_only"},{"task":"embed"}]:
    print(j, "->", route_model(j))


In [ ]:
# Demo 4: bakeoff harness
from statistics import mean
bench = [
    {"gold":"19.99","preds":{"mini":"19.99","frontier":"19.99"}},
    {"gold":"ACME","preds":{"mini":"ACME INC","frontier":"ACME"}},
    {"gold":"2026-07-01","preds":{"mini":"2026-07-01","frontier":"2026-07-01"}},
]
for m in ("mini","frontier"):
    print(m, round(mean(p[m].lower()==row["gold"].lower() for row in bench for p in [row["preds"]]),3))


In [ ]:
# Demo 5: cost model $/1K images
def cost_per_1k(tokens_per_image, price_per_m_tokens, images=1000):
    return images * tokens_per_image / 1e6 * price_per_m_tokens
print("mini", cost_per_1k(800, 0.15))
print("frontier", cost_per_1k(1200, 2.5))


In [ ]:
# Demo 6: Anthropic-style image content block shape (illustrative)
import json
anthropic_msg = {
    "model": "claude-sonnet-4-YOUR_MODEL",
    "max_tokens": 300,
    "messages": [{"role":"user","content":[
        {"type":"image","source":{"type":"base64","media_type":"image/png","data":"YOUR_BASE64_IMAGE"}},
        {"type":"text","text":"Extract the table as CSV."},
    ]}],
}
print(json.dumps(anthropic_msg, indent=2)[:280])
print("ANTHROPIC_API_KEY=YOUR_ANTHROPIC_API_KEY")


### Specialized vs generalist

| Need | Prefer |
|------|--------|
| Alt-text volume | Mini / distilled |
| Dense doc JSON | Doc-strong + OCR hybrid |
| Image search | Dual-encoder |
| UI agents | Screenshot-tuned + tools |
| Air-gapped | Open weights in VPC |
| Medical/legal | Domain + HITL |


### Checklist — Vendor due diligence

- [ ] Image retention & training-use terms
- [ ] Region/residency options
- [ ] Rate limits / max images
- [ ] Structured output support
- [ ] Incident / abuse contact


### Try it yourself — Selection

1. Add latency_sla_ms to router; avoid frontier if SLA < 800ms.
2. Soft-match vendor names in bakeoff.
3. Write 5 retention due-diligence questions.

**Stretch:** Price 1M pages/month in a tiny spreadsheet function.


### Try it yourself — Multi-provider

1. Normalize OpenAI and Anthropic message builders behind one interface.
2. Add fallback if primary returns 429.


## Knowledge Check

**Q1.** When is self-host worth it?

<details><summary>Answer</summary>

Air-gap, strict residency, predictable high volume where GPU cost < API margin.

</details>

**Q2.** Why freeze a bakeoff set?

<details><summary>Answer</summary>

Vendor silent updates and prompt tweaks otherwise make quality undebuggable.

</details>


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `bakeoff` | Frozen eval on your data |
| `residency` | Where data may be processed |
| `open weights` | Downloadable parameters |
| `router` | Per-request model policy |
| `specialized VLM` | Narrow-domain MM model |


## Key Takeaways

- Hosted, regional, open, specialized serve different constraints
- Score multi-objective — not one leaderboard
- Route easy/hard; escalate on validation failure
- Frozen bakeoffs catch regressions


## Production Incident Patterns — VLM model selection

| Symptom | Likely cause | First fix |
|---------|--------------|-----------|
| Sudden cost spike | `detail=high` on huge pages | Resize + tile budget |
| Fluent wrong fields | VLM hallucination | OCR hybrid + schema |
| Cross-customer leak | Missing tenant filter | ACL in retriever code |
| Flaky eval scores | Unfrozen prompts/models | Pin versions + bakeoff set |
| Latency SLO burn | Full-page high detail | Crop ROI → mini model |

```
ASCII control loop:
  ingest -> normalize -> route model -> generate -> validate -> (HITL|export)
                     ^                              |
                     +-------- metrics/audit <------+
```


In [ ]:
# Cross-cutting: redact secrets before logging multimodal payloads
import re, json

SECRET_RE = re.compile(r"(api[_-]?key|bearer\s+[A-Za-z0-9._\-]+)", re.I)

def safe_log(payload: dict) -> str:
    s = json.dumps(payload)
    s = SECRET_RE.sub("***", s)
    if "base64," in s:
        s = re.sub(r"base64,[A-Za-z0-9+/=]+", "base64,[REDACTED]", s)
    return s[:500]

print(safe_log({
    "model": "gpt-4o",
    "api_key": "YOUR_OPENAI_API_KEY",
    "content": "data:image/png;base64,AAAABBBBCCCC",
    "topic": "VLM model selection",
}))


## Mini Case Study — VLM model selection

**Scenario:** A team ships a vision feature in one week. Demo looks great on three happy-path images.
**Week 2:** finance reports wrong totals; legal asks about image retention; GPU/API bill 4× forecast.

**Retro questions**
1. What was the output contract (schema) on day one?
2. Which failure mode had no metric?
3. Was there a crop/detail budget?
4. Who owns HITL and appeals?

**Design rule:** if a field can move money or identity, it needs a validator + disagreement path before automation.


In [ ]:
# Cross-cutting: simple SLO helper for vision endpoints
from dataclasses import dataclass

@dataclass
class VisionSLO:
    availability: float = 0.995
    p95_ms: int = 4000
    max_critical_field_error_rate: float = 0.005

def breached(slo: VisionSLO, avail: float, p95: int, crit_err: float) -> list[str]:
    out = []
    if avail < slo.availability: out.append("availability")
    if p95 > slo.p95_ms: out.append("latency")
    if crit_err > slo.max_critical_field_error_rate: out.append("critical_accuracy")
    return out or ["ok"]

print("VLM model selection", breached(VisionSLO(), 0.99, 5200, 0.02))


### Try it yourself — VLM model selection ops

1. Write a one-page runbook section for on-call when VLM model selection critical_accuracy SLO breaches.
2. Add a dashboard sketch: cost/1k images, CER/field error, HITL rate, p95 latency.
